## Project initialization

Create a project: a dedicated space where we can manage functions, artifacts and executions.

Use the ``username`` as param to create a personal project in the shared space.

In [ ]:
import digitalhub as dh
import os

project = dh.get_or_create_project(f"{os.environ['USER']}-base")
project

### Get a function run and invoke the service

We can look up the run by getting the function or the id.

In [ ]:
fn = project.get_function("echoservice")

In [ ]:
run = fn.list_runs()[0]

In [ ]:
run.status.service

With the service details we can invoke it: let's pick the URL and make a POST

In [ ]:
URL=run.status.service["url"]

In [ ]:
# invoke manually
import requests
import pprint
pp = pprint.PrettyPrinter(indent=2)

res = requests.post(f"http://{URL}",json={"text": f"{os.environ['USER']}-base"})
pp.pprint(res.json())

calling a service is a standard operation, we can leverage the SDK


In [ ]:
run.invoke(json={"text": f"{os.environ['USER']}-base"}).json()


## Write a custom service

We can define our own custom service: write a function and then launch a serve run.
Every time the server receives an HTTP call, our function will receive the event.


So let's write a file and then define a function with the file as source

In [ ]:
%%writefile "helloservice.py"

import json


def serve(context, event):

    if isinstance(event.body, bytes):
        body = json.loads(event.body)
    else:
        body = event.body
    context.logger.info(f"Received event: {body}")

    return {"message": f"Hello, {body['name']}!"}

In [ ]:
func = project.new_function(name="hello-service",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="helloservice.py",
                            handler="serve")

Now we should be able to run it locally (`local_execution=True`). Or not?

In [ ]:
run = func.run("job", local_execution=True)

Run properly and then try invoking the service.
Check the logs as well.